<a href="https://colab.research.google.com/github/vigneshvicky0246/inlighn-tech/blob/main/Predict_Customer_Churn_with_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Predict Customer Churn with Python**



In [1]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV , KFold , cross_val_score

In [2]:
# Classification algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis , QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb
import xgboost as xgb


In [3]:
# Classification metrices
from sklearn.metrics import accuracy_score,confusion_matrix, classification_report, precision_score,recall_score,f1_score

## **3. Read dataset**

In [ ]:
from google.colab import drive

drive.mount('/content/drive')



In [ ]:
file_path = '/content/drive/MyDrive/datasets/Customer Churn/Churn_Modelling.csv'


In [ ]:
df = pd.read_csv(file_path)

## **4. Exploratory Data Analysis**

- Now, we will perform EDA to gain insights about our data.

### **4.1 Shape of dataset**

In [ ]:
df.shape

- Our dataset contains 10000 instances and 14 variables.

- Now, let's take a quick look of our dataset.

### **4.2 Preview dataset**

In [ ]:
df.head()

### **4.3 Summary of dataset**

In [ ]:
df.info()

- We can see that there are 3 categorical variables and 11 numerical variables in the dataset.

- Also, there are no missing values in the dataset.

### **4.4 Statistical properties of dataset**

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)


In [ ]:
df.columns

- We can see that the above three columns are removed from the dataset.

## **6. Convert categorical columns to numeric columns**


- Now, in our dataset, we have two categorical columns: **Gender** and **Geography**. We should convert them into numerical format.

#### **6.1 Explore Gender variable**

In [ ]:
df['Gender'].value_counts()

In [ ]:
# let's do One Hot Encoding of Gender variable
# get k dummy variables after One Hot Encoding


df['Gender'] = pd.get_dummies(df['Gender'], drop_first = False)

- Now, let's take a look again at our dataset.

In [ ]:
df.head()

- Here 1 stands for female and 0 stands for male.

#### **6.2 Explore Geography variable**

In [ ]:
df['Geography'].value_counts()

In [ ]:
# let's do One Hot Encoding of Goegraphy variable
# get k dummy variables after One Hot Encoding


df['Geography'] = pd.get_dummies(df['Geography'], drop_first = False)



- Let's again take a look at our dataset.

In [ ]:
df.head()

## **7. Feature Scaling**

In [ ]:
X =  df.drop(['Exited'], axis=1)
y = df['Exited']


In [ ]:
cols = X.columns

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(X)

In [ ]:
X = pd.DataFrame(X, columns=[cols])

In [ ]:
X.head()

## **8. Model Training**

- We can see that our dataset is now ready to be fed into a ML model. We will proceed as follows:-


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

#### **8.1 Predict accuracy with different algorithms**


- Let's predict accuracy with different algorithms and evaluate their performance.

In [ ]:
names = ["Logistic Regression", "Nearest Neighbors", "Naive Bayes", "Linear SVM", "RBF SVM",
         "Decision Tree", "Random Forest", "AdaBoost", "Gradient Boosting",
         "LDA", "QDA", "Neural Net", "XGBoost" ]

In [ ]:
classifiers = [
    LogisticRegression(),
    KNeighborsClassifier(5),
    GaussianNB(),
    SVC(kernel="linear", C=0.025),
    SVC(kernel = "rbf", gamma=2, C=1),
    DecisionTreeClassifier(max_depth=5),
    RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1),
    AdaBoostClassifier(),
    GradientBoostingClassifier(),
    LinearDiscriminantAnalysis(),
    QuadraticDiscriminantAnalysis(),
    MLPClassifier(alpha=1, max_iter=1000),
    xgb.XGBClassifier()
   ]

In [ ]:
accuracy_scores = []

# iterate over classifiers and predict accuracy
for name, clf in zip(names, classifiers):
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    score = round(score, 4)
    accuracy_scores.append(score)
    print(name ,' : ' , score)

In [ ]:
classifiers_performance = pd.DataFrame({"Classifiers": names, "Accuracy Scores": accuracy_scores})
classifiers_performance

- The accuracy score of top performing algorithms in descending order is given below -

In [ ]:
classifiers_performance.sort_values(by = 'Accuracy Scores' , ascending = False)[['Classifiers', 'Accuracy Scores']]

#### **8.2 Plot the classifier accuracy scores**

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
x = classifiers_performance['Accuracy Scores']
y = classifiers_performance['Classifiers']
ax.barh(y, x, align='center', color='green')
ax.invert_yaxis()  # labels read top-to-bottom
ax.set_xlabel('Accuracy Scores')
ax.set_ylabel('Classifiers', rotation=0)
ax.set_title('Classifier Accuracy Scores')
plt.show()

## **9. Feature Importance**

- In this section, we will see how to improve model performance by feature selection.

- We will visualize feature importance with random forest classifier and drop the least important feature, rebuild the model and check effect on accuracy.

- For a comprehensive overview on feature selection techniques, please see the kernel -

[A Reference Guide to Feature Selection Methods](https://www.kaggle.com/prashant111/comprehensive-guide-on-feature-selection?scriptVersionId=47174422)

#### **9.1 Feature importance with Random Forest model**


- Until now, I have used all the features given in the model. Now, I will select only the important features, build the model using these features and see its effect on accuracy.

- First, I will create the Random Forest model as follows:-

In [ ]:
# instantiate the classifier with n_estimators = 100
clf = RandomForestClassifier(n_estimators=100, random_state=0)


# fit the classifier to the training set
clf.fit(X_train, y_train)

- Now, I will use the feature importance variable to see feature importance scores.

In [ ]:
# view the feature scores
feature_scores = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feature_scores

- We can see that the most important feature is **Age** and least important feature is **Gender**.

#### **9.2 Drop least important feature**

- Now, I will drop the least important feature **Gender** from the model, rebuild the model and check its effect on accuracy.

In [ ]:
# drop the least important feature Gender from X_train and X_test for further analysis
X1_train = X_train.drop(['Gender'], axis=1)
X1_test = X_test.drop(['Gender'], axis=1)

In [ ]:
accuracy_scores1 = []

# iterate over classifiers and predict accuracy
for name, clf in zip(names, classifiers):
    clf.fit(X1_train, y_train)
    score = clf.score(X1_test, y_test)
    score = round(score, 4)
    accuracy_scores1.append(score)
    print(name ,' : ' , score)

- Now, we will compare our original accuracy and latest accuracy from the model.

In [ ]:
classifiers_performance1 = pd.DataFrame({"Classifiers": names, "Accuracy Scores": accuracy_scores,
                                         "Accuracy Scores1": accuracy_scores1})
classifiers_performance1

- We can see that dropping the least important feature **Gender** from the model does not result in performance improvement.

- We can see that XGBoost has maximum accuracy of 0.866.

- So, we will use the XGBoost Classifier to plot the confusion-matrix.

## **10. Confusion Matrix**


- A **confusion matrix** is a tool for summarizing the performance of a classification algorithm. A confusion matrix will give us a clear picture of classification model performance and the types of errors produced by the model. It gives us a summary of correct and incorrect predictions broken down by each category. The summary is represented in a tabular form.

- Four types of outcomes are possible while evaluating a classification model performance. These four outcomes are described below:-

- **True Positives (TP)** – True Positives occur when we predict an observation belongs to a certain class and the observation actually belongs to that class.

- **True Negatives (TN)** – True Negatives occur when we predict an observation does not belong to a certain class and the observation actually does not belong to that class.

- **False Positives (FP)** – False Positives occur when we predict an observation belongs to a certain class but the observation actually does not belong to that class. This type of error is called Type I error.

- **False Negatives (FN)** – False Negatives occur when we predict an observation does not belong to a certain class but the observation actually belongs to that class. This is a very serious error and it is called Type II error.

- These four outcomes are summarized in a confusion matrix.

- We will use the XGBoost Classifier to plot the confusion-matrix.

In [ ]:
# instantiate the XGBoost classifier
xgb_clf = xgb.XGBClassifier()


# fit the classifier to the modified training set
xgb_clf.fit(X_train, y_train)

In [ ]:
# predict on the test set
y_pred = xgb_clf.predict(X_test)

In [ ]:
# print the accuracy
print('XGBoost Classifier model accuracy score: {0:0.4f}'. format(accuracy_score(y_test, y_pred)))

In [ ]:
# print confusion-matrix

cm = confusion_matrix(y_test, y_pred)

print('Confusion matrix\n\n', cm)

print('\nTrue Positives(TP) = ', cm[0,0])

print('\nTrue Negatives(TN) = ', cm[1,1])

print('\nFalse Positives(FP) = ', cm[0,1])

print('\nFalse Negatives(FN) = ', cm[1,0])

The confusion matrix shows 1557 + 175 = 1732 correct predictions and 50 + 218 = 268 incorrect predictions.

In this case, we have

- True Positives (Actual Positive:1 and Predict Positive:1) - 1557
- True Negatives (Actual Negative:0 and Predict Negative:0) - 175
- False Positives (Actual Negative:0 but Predict Positive:1) - 50 (Type I error)
- False Negatives (Actual Positive:1 but Predict Negative:0) - 218 (Type II error)

In [ ]:
# visualize confusion matrix with seaborn heatmap

cm_matrix = pd.DataFrame(data=cm, columns=['Actual Positive:1', 'Actual Negative:0'],
                                 index=['Predict Positive:1', 'Predict Negative:0'])

sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='YlGnBu')

## **11. Classification Metrices**

#### **11.1 Classification Report**


- **Classification Report** is another way to evaluate the classification model performance.
- It displays the **precision**, **recall**, **f1** and **support** scores for the model.
- We can print a classification report as follows:-

In [ ]:
print(classification_report(y_test, y_pred))

#### **11.2 Classification Accuracy**

In [ ]:
TP = cm[0,0]
TN = cm[1,1]
FP = cm[0,1]
FN = cm[1,0]

In [ ]:
# print classification accuracy

classification_accuracy = (TP + TN) / float(TP + TN + FP + FN)

print('Classification accuracy : {0:0.4f}'.format(classification_accuracy))

#### **11.3 Classification Error**

In [ ]:
# print classification error

classification_error = (FP + FN) / float(TP + TN + FP + FN)

print('Classification error : {0:0.4f}'.format(classification_error))

#### **11.4 Precision**

- **Precision** can be defined as the percentage of correctly predicted positive outcomes out of all the predicted positive outcomes. It can be given as the ratio of true positives (TP) to the sum of true and false positives (TP + FP).

- So, Precision identifies the proportion of correctly predicted positive outcome. It is more concerned with the positive class than the negative class.

- Mathematically, precision can be defined as the ratio of TP to (TP + FP).

In [ ]:
# print precision score

precision = TP / float(TP + FP)

print('Precision : {0:0.4f}'.format(precision))

#### **11.5 Recall**

- **Recall** can be defined as the percentage of correctly predicted positive outcomes out of all the actual positive outcomes. It can be given as the ratio of true positives (TP) to the sum of true positives and false negatives (TP + FN).

- **Recall** is also called **Sensitivity**.

- Recall identifies the proportion of correctly predicted actual positives.

- Mathematically, Recall can be given as the ratio of TP to (TP + FN).

In [ ]:
recall = TP / float(TP + FN)

print('Recall or Sensitivity : {0:0.4f}'.format(recall))

#### **11.6 True Positive Rate**

- **True Positive Rate** is synonymous with **Recall**.

In [ ]:
true_positive_rate = TP / float(TP + FN)

print('True Positive Rate : {0:0.4f}'.format(true_positive_rate))

#### **11.7 False Positive Rate**

In [ ]:
false_positive_rate = FP / float(FP + TN)

print('False Positive Rate : {0:0.4f}'.format(false_positive_rate))


#### **11.8 Specificity (True Negative Rate)**


- **Specificity** is also called **True Negative Rate**.

In [ ]:
specificity = TN / (TN + FP)

print('Specificity : {0:0.4f}'.format(specificity))

#### **11.9 f1-score**

- **f1-score** is the weighted harmonic mean of precision and recall.
- The best possible f1-score would be 1.0 and the worst would be 0.0.
- f1-score is the harmonic mean of precision and recall.
- So, f1-score is always lower than accuracy measures as they embed precision and recall into their computation.
- The weighted average of f1-score should be used to compare classifier models, not global accuracy.

#### **11.10 Support**

- **Support** is the actual number of occurrences of the class in our dataset.

## **12. Cross Validation**


- We will check whether **Cross Validation** results in performance improvement.

In [ ]:
# iterate over classifiers and calculate cross-validation score
for name, clf in zip(names, classifiers):
    scores = cross_val_score(clf, X_train, y_train, cv = 10, scoring='accuracy')
    print(name , ':{:.4f}'.format(scores.mean()))